# LoRA-mBART50 End-to-End Sichuanese -> English
#
Your setting:
- You are responsible for one model: LoRA-mBART50 end-to-end `SC -> EN`.
- `train_9800_zh.csv` has no English column, only:
  - `text`: Sichuanese source
  - `mandarin`: Mandarin pivot
- `test_200_en.csv` has the 200-row test set with gold/reference English.
#
Pipeline:
1. Use NLLB to translate train Mandarin -> pseudo English.
2. Fine-tune LoRA-mBART50 on `text -> pseudo_english`.
3. Run LoRA-mBART50 on test `text -> pred_english`.
4. Evaluate against test gold English with BLEU and chrF.
#
Kaggle input:
- Upload `train_9800_zh.csv` and `test_200_en.csv` to one Kaggle Dataset.
- Attach that Dataset to this notebook.
- Enable GPU and Internet.
#
Outputs:
- `/kaggle/working/train_9800_zh_pseudo_en.csv`
- `/kaggle/working/lora_mbart_sc_en_predictions.csv`
- `/kaggle/working/lora_mbart_sc_en_metrics.csv`
- `/kaggle/working/lora_mbart_sc_en_cases.csv`
- `/kaggle/working/mbart_lora_sc_en_adapter/`

## 0. Install Packages

In [1]:
!pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import os
import re
import sys
import glob
import random
import subprocess
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

INSTALL_PACKAGES = True

if INSTALL_PACKAGES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "transformers",
            "datasets",
            "accelerate",
            "peft",
            "sacrebleu",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )

In [3]:
import numpy as np
import pandas as pd
import torch
import sacrebleu
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

## 1. Config

In [4]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = "/kaggle/input"
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Leave these as None for filename auto-detection under /kaggle/input.
TRAIN_CSV = None  # expected file: train_9800_zh.csv
TEST_CSV = None   # expected file: test_200_en.csv

# Confirmed column names in the provided files.
SC_COL = "text"
ZH_COL = "mandarin"
GOLD_EN_COL = "english"

PSEUDO_EN_COL = "pseudo_english_nllb"
PRED_EN_COL = "pred_english_lora_mbart"

# Use 600M if Kaggle memory is tight. Use 1.3B if you have enough GPU memory.
NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"
MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

NLLB_ZH_LANG = "zho_Hans"
NLLB_EN_LANG = "eng_Latn"
MBART_SRC_LANG = "zh_CN"
MBART_TGT_LANG = "en_XX"

MAX_SOURCE_LEN = 96
MAX_TARGET_LEN = 128
MAX_NEW_TOKENS_EN = 128

NLLB_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-4
DEV_SIZE = 500

# For a smoke test before full run:
# TRAIN_LIMIT = 300
# TEST_LIMIT = 30
TRAIN_LIMIT = None
TEST_LIMIT = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print("Device:", DEVICE)

Device: cuda


## 2. Load CSVs

In [5]:
def find_csv(preferred_names, required=True):
    csvs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "*.csv"), recursive=True))
    for preferred in preferred_names:
        matches = [p for p in csvs if preferred.lower() in os.path.basename(p).lower()]
        if matches:
            return sorted(matches, key=len)[0]
    if required:
        raise FileNotFoundError(
            f"Could not find any of {preferred_names} under {INPUT_ROOT}. "
            f"Found CSVs: {csvs[:30]}"
        )
    return None


def read_csv_flexible(path):
    for enc in ["utf-8-sig", "utf-8", "gb18030"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


if TRAIN_CSV is None:
    TRAIN_CSV = find_csv(["train_9800_zh", "train_9800_mandarin", "train_9800"])
if TEST_CSV is None:
    TEST_CSV = find_csv(["test_200_en", "tr_test_200_en", "test_200"])

train_df = read_csv_flexible(TRAIN_CSV)
test_df = read_csv_flexible(TEST_CSV)

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("Train:", train_df.shape, list(train_df.columns))
print("Test: ", test_df.shape, list(test_df.columns))
display(train_df.head(3))
display(test_df.head(3))

TRAIN_CSV: /kaggle/input/datasets/ousyuwang/sichuan-mt-data/train_9800_zh.csv
TEST_CSV: /kaggle/input/datasets/ousyuwang/sichuan-mt-data/test_200_en.csv
Train: (9800, 6) ['source_file', 'utt_or_id', 'filename', 'text', 'matched_terms', 'mandarin']
Test:  (200, 7) ['source_file', 'utt_or_id', 'filename', 'text', 'matched_terms', 'mandarin', 'english']


,source_file,utt_or_id,filename,text,matched_terms,mandarin
0,wsc_train_audio_text.tsv.gz,cq00015015_65860_81310,cq00015015_65860_81310.wav,老房子家具家电齐全直接就可以拎包入住这个位置这个户型这个装修你们喜欢吗喜欢的话记到联系我哟上...,你们 这个 你 我,老房子家具家电齐全，可以直接拎包入住。这个位置、这个户型、这个装修，你们喜欢吗？如果喜欢的话...
1,wsc_train_audio_text.tsv.gz,cq00009746_159450_171270,cq00009746_159450_171270.wav,反正他车在那里放着也是放着租不出去也损失了那么我粉丝比如说正常正常a六该住六百一天的该住五百...,粉丝 我 他,反正他的车放在那里也是放着，租不出去也会有损失。比如我的粉丝，比如说正常的A6应该住600一...
2,wsc_train_audio_text.tsv.gz,cq00001785_7080_20010,cq00001785_7080_20010.wav,客户直接问一下那是你们幺儿吗你说我哪儿像那长得像那个同事他们妈咹我到底比他老好多你们说我看起...,他们 同事 那个 幺儿 你们 你 我 他 妈,客户直接问一下，那是你们的孩子吗？我说我哪儿像呢，长得不像那个同事他们妈妈呢？我到底比他大好...


,source_file,utt_or_id,filename,text,matched_terms,mandarin,english
0,wsc_train_audio_text.tsv.gz,704219202372_Al7Ir_22_3640,704219202372_Al7Ir_22_3640.wav,我是捡镜子的命法莫法生了这个命哦命哦,这个 我,我这是捡镜子的命，没办法生成这样的命啊。,"This is the fate of the mirror, there is no wa..."
1,wsc_train_audio_text.tsv.gz,013165493347_ECZsx_44_7720,013165493347_ECZsx_44_7720.wav,妈,妈,妈妈,It's called
2,wsc_train_audio_text.tsv.gz,cq0034024_86130_88550,cq0034024_86130_88550.wav,谢谢谢谢李婆婆,婆婆,谢谢谢谢李奶奶,Thank you very much 李奶奶


In [6]:
def norm_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_")


def guess_col(df, candidates, role, required=True):
    if role == "sc" and SC_COL is not None and SC_COL in df.columns:
        return SC_COL
    if role == "zh" and ZH_COL is not None and ZH_COL in df.columns:
        return ZH_COL
    if role == "en" and GOLD_EN_COL is not None and GOLD_EN_COL in df.columns:
        return GOLD_EN_COL

    normalized = {norm_name(c): c for c in df.columns}
    for cand in candidates:
        cand_norm = norm_name(cand)
        for n, original in normalized.items():
            if cand_norm == n or cand_norm in n:
                return original
    if required:
        raise ValueError(
            f"Could not infer {role} column. Columns: {list(df.columns)}. "
            "Set SC_COL / ZH_COL / GOLD_EN_COL manually in the config cell."
        )
    return None


SC_CANDIDATES = [
    "text",
    "sc",
    "sichuanese",
    "sichuan",
    "dialect",
    "source",
    "src",
    "四川话",
    "川话",
    "方言",
    "原文",
]
ZH_CANDIDATES = [
    "mandarin",
    "zh",
    "chinese",
    "pivot",
    "normalization",
    "普通话",
    "中文",
    "中介",
]
EN_CANDIDATES = [
    "gold_english",
    "reference_english",
    "ref_english",
    "gold_en",
    "english",
    "en",
    "target",
    "translation",
    "英文",
    "英语",
]

TRAIN_SC_COL = guess_col(train_df, SC_CANDIDATES, "sc")
TRAIN_ZH_COL = guess_col(train_df, ZH_CANDIDATES, "zh")
TEST_SC_COL = guess_col(test_df, SC_CANDIDATES, "sc")
TEST_EN_COL = guess_col(test_df, EN_CANDIDATES, "en")

print("Detected columns:")
print("  train SC:", TRAIN_SC_COL)
print("  train ZH:", TRAIN_ZH_COL)
print("  test SC: ", TEST_SC_COL)
print("  test EN: ", TEST_EN_COL)

Detected columns:
  train SC: text
  train ZH: mandarin
  test SC:  text
  test EN:  english


## 3. Generate Pseudo English for Train with NLLB

In [7]:
def clean_text_series(s):
    return s.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})


train_df[TRAIN_SC_COL] = clean_text_series(train_df[TRAIN_SC_COL])
train_df[TRAIN_ZH_COL] = clean_text_series(train_df[TRAIN_ZH_COL])
train_df = train_df.dropna(subset=[TRAIN_SC_COL, TRAIN_ZH_COL]).reset_index(drop=True)
if TRAIN_LIMIT is not None:
    train_df = train_df.head(TRAIN_LIMIT).copy()

test_df[TEST_SC_COL] = clean_text_series(test_df[TEST_SC_COL])
test_df[TEST_EN_COL] = clean_text_series(test_df[TEST_EN_COL])
test_df = test_df.dropna(subset=[TEST_SC_COL, TEST_EN_COL]).reset_index(drop=True)
if TEST_LIMIT is not None:
    test_df = test_df.head(TEST_LIMIT).copy()

print("Clean train:", train_df.shape)
print("Clean test: ", test_df.shape)

Clean train: (9800, 6)
Clean test:  (200, 7)


In [8]:
def batch_iter(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)
nllb_model.eval()


@torch.inference_mode()
def nllb_zh_to_en(texts, batch_size=NLLB_BATCH_SIZE):
    nllb_tokenizer.src_lang = NLLB_ZH_LANG
    tgt_lang_id = nllb_tokenizer.convert_tokens_to_ids(NLLB_EN_LANG)
    outputs = []
    texts = [str(x).strip() for x in texts]
    for batch in batch_iter(texts, batch_size):
        inputs = nllb_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(DEVICE)
        generated = nllb_model.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_new_tokens=MAX_NEW_TOKENS_EN,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
        outputs.extend(nllb_tokenizer.batch_decode(generated, skip_special_tokens=True))
        print(f"Generated pseudo EN: {len(outputs)}/{len(texts)}", end="\r")
    print()
    return [x.strip() for x in outputs]


if PSEUDO_EN_COL not in train_df.columns:
    train_df[PSEUDO_EN_COL] = nllb_zh_to_en(train_df[TRAIN_ZH_COL].tolist())
else:
    missing_mask = train_df[PSEUDO_EN_COL].isna() | (train_df[PSEUDO_EN_COL].astype(str).str.strip() == "")
    if missing_mask.any():
        train_df.loc[missing_mask, PSEUDO_EN_COL] = nllb_zh_to_en(
            train_df.loc[missing_mask, TRAIN_ZH_COL].tolist()
        )

pseudo_path = WORK_DIR / "train_9800_zh_pseudo_en.csv"
train_df.to_csv(pseudo_path, index=False, encoding="utf-8-sig")
print("Saved pseudo-label train file:", pseudo_path)
display(train_df[[TRAIN_SC_COL, TRAIN_ZH_COL, PSEUDO_EN_COL]].head(5))

# Free GPU memory before mBART training.
nllb_model.to("cpu")
del nllb_model
torch.cuda.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated pseudo EN: 9800/9800
Saved pseudo-label train file: /kaggle/working/train_9800_zh_pseudo_en.csv


,text,mandarin,pseudo_english_nllb
0,老房子家具家电齐全直接就可以拎包入住这个位置这个户型这个装修你们喜欢吗喜欢的话记到联系我哟上...,老房子家具家电齐全，可以直接拎包入住。这个位置、这个户型、这个装修，你们喜欢吗？如果喜欢的话...,The old house furniture is full of electricity...
1,反正他车在那里放着也是放着租不出去也损失了那么我粉丝比如说正常正常a六该住六百一天的该住五百...,反正他的车放在那里也是放着，租不出去也会有损失。比如我的粉丝，比如说正常的A6应该住600一...,"Instead of his car being left there, it's also..."
2,客户直接问一下那是你们幺儿吗你说我哪儿像那长得像那个同事他们妈咹我到底比他老好多你们说我看起...,客户直接问一下，那是你们的孩子吗？我说我哪儿像呢，长得不像那个同事他们妈妈呢？我到底比他大好...,"If the client asks you directly, is that your ..."
3,你们几点钟开到几点钟啊,你们几点开始开到几点啊,What time do you start at?
4,哦个人嗯不行嗯,这个人不行了,This guy didn't walk.


## 4. Fine-tune LoRA-mBART50: Sichuanese -> English

In [9]:
train_pairs = train_df[[TRAIN_SC_COL, PSEUDO_EN_COL]].copy()
train_pairs.columns = ["src", "tgt"]
train_pairs["src"] = clean_text_series(train_pairs["src"])
train_pairs["tgt"] = clean_text_series(train_pairs["tgt"])
train_pairs = train_pairs.dropna(subset=["src", "tgt"]).drop_duplicates().reset_index(drop=True)

dev_size = min(DEV_SIZE, max(1, int(0.1 * len(train_pairs))))
shuffled = train_pairs.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
dev_pairs = shuffled.iloc[:dev_size].reset_index(drop=True)
fit_pairs = shuffled.iloc[dev_size:].reset_index(drop=True)

print("Fit pairs:", fit_pairs.shape)
print("Dev pairs:", dev_pairs.shape)

Fit pairs: (9235, 2)
Dev pairs: (500, 2)


In [10]:
mbart_tokenizer = AutoTokenizer.from_pretrained(MBART_MODEL_NAME)
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(MBART_MODEL_NAME, torch_dtype=DTYPE)

mbart_tokenizer.src_lang = MBART_SRC_LANG
mbart_tokenizer.tgt_lang = MBART_TGT_LANG
mbart_model.config.decoder_start_token_id = mbart_tokenizer.lang_code_to_id[MBART_TGT_LANG]
mbart_model.config.forced_bos_token_id = mbart_tokenizer.lang_code_to_id[MBART_TGT_LANG]

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
mbart_model = get_peft_model(mbart_model, lora_config)
mbart_model.print_trainable_parameters()

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 613,238,784 || trainable%: 0.3847


In [11]:
fit_ds = Dataset.from_pandas(fit_pairs, preserve_index=False)
dev_ds = Dataset.from_pandas(dev_pairs, preserve_index=False)


def preprocess_mbart(batch):
    mbart_tokenizer.src_lang = MBART_SRC_LANG
    mbart_tokenizer.tgt_lang = MBART_TGT_LANG
    inputs = mbart_tokenizer(
        batch["src"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    labels = mbart_tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    inputs["labels"] = labels["input_ids"]
    return inputs


fit_tok = fit_ds.map(preprocess_mbart, batched=True, remove_columns=fit_ds.column_names)
dev_tok = dev_ds.map(preprocess_mbart, batched=True, remove_columns=dev_ds.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=mbart_tokenizer,
    model=mbart_model,
    label_pad_token_id=-100,
)

Map:   0%|          | 0/9235 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [13]:
training_args_kwargs = dict(
    output_dir=str(WORK_DIR / "mbart_lora_sc_en_checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_steps=200,
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=False,
)

try:
    training_args = Seq2SeqTrainingArguments(
        **training_args_kwargs,
        evaluation_strategy="steps",
    )
except TypeError:
    training_args = Seq2SeqTrainingArguments(
        **training_args_kwargs,
        eval_strategy="steps",
    )

trainer_kwargs = dict(
    model=mbart_model,
    args=training_args,
    train_dataset=fit_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
)

try:
    trainer = Seq2SeqTrainer(
        **trainer_kwargs,
        processing_class=mbart_tokenizer,
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        **trainer_kwargs,
        tokenizer=mbart_tokenizer,
    )

trainer.train()


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
200,12.207441,3.007812
400,11.976875,2.970703
578,11.696113,2.958984


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=578, training_loss=12.159422983347751, metrics={'train_runtime': 1073.0542, 'train_samples_per_second': 17.213, 'train_steps_per_second': 0.539, 'total_flos': 2181689197363200.0, 'train_loss': 12.159422983347751, 'epoch': 2.0})

## 5. Predict English on Test 200

In [16]:
@torch.inference_mode()
def generate_mbart_en(texts, batch_size=EVAL_BATCH_SIZE):
    mbart_model.eval()
    mbart_model.to(DEVICE)
    mbart_tokenizer.src_lang = MBART_SRC_LANG

    # Clear legacy generation settings from model.config.
    mbart_model.config.forced_bos_token_id = None
    mbart_model.config.decoder_start_token_id = None

    # Use the new generation_config API.
    mbart_model.generation_config.decoder_start_token_id = mbart_tokenizer.lang_code_to_id[MBART_TGT_LANG]
    mbart_model.generation_config.forced_bos_token_id = mbart_tokenizer.lang_code_to_id[MBART_TGT_LANG]

    outputs = []
    texts = [str(x).strip() for x in texts]

    for batch in batch_iter(texts, batch_size):
        inputs = mbart_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(DEVICE)

        generated = mbart_model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

        outputs.extend(mbart_tokenizer.batch_decode(generated, skip_special_tokens=True))
        print(f"Predicted test EN: {len(outputs)}/{len(texts)}", end="\r")

    print()
    return [x.strip() for x in outputs]


test_df[PRED_EN_COL] = generate_mbart_en(test_df[TEST_SC_COL].tolist())
display(test_df[[TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].head(10))


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted test EN: 200/200


,text,english,pred_english_lora_mbart
0,我是捡镜子的命法莫法生了这个命哦命哦,"This is the fate of the mirror, there is no wa...","I'm picking up a mirror, and I can't live this..."
1,妈,It's called,Mom
2,谢谢谢谢李婆婆,Thank you very much 李奶奶,"Thank you, thank you."
3,我有几个朋友都强烈推荐他们家的小面说海椒香得很点一碗干的大款来搞哈调料就是这几种也不复杂没得...,Several of my friends have strongly recommende...,I have a few friends who strongly recommend th...
4,完全由个人的爱好来决定你可以两样都做一次,"purely based on personal interest, you can try...","It's all about your personal interests, and yo..."
5,快要个en了我们要走了明要我可以看我可朋友明我朋友们的,"We're almost there, we're ready to go.Tomorrow...",We're going to see my friends.
6,真的还是碰对了伸手又把三筒摸回来这一下这个痛青就稳当了,"Even if it's a real hit, reaching out and touc...","It's really hard to touch it again, and it fee..."
7,给大家呢就是嗯普及一些知识就想说一下四川目前的民用机场那大家晓得哈四川航空呢其实是一个非常优...,"To spread some knowledge, I want to talk about...",I want to share some common knowledge with you...
8,但曰天下人民欲时时常事之肉陛下禁止如父母生物寸功皆封裂侯矿风虚等接连黄金欲为内乱陛下禁不自省...,But if the people of this world want to partic...,But he said that the people of the world desir...
9,那个时候大人忙农活和家务孩子呢不仅打不上手,At that time adults were so busy with farming ...,"At that time, adults were busy with farming an..."


## 6. Evaluate BLEU and chrF

In [17]:
def corpus_scores(preds, refs):
    preds = [str(x).strip() for x in preds]
    refs = [str(x).strip() for x in refs]
    return {
        "BLEU": sacrebleu.corpus_bleu(preds, [refs]).score,
        "chrF": sacrebleu.corpus_chrf(preds, [refs]).score,
    }


scores = corpus_scores(test_df[PRED_EN_COL].tolist(), test_df[TEST_EN_COL].tolist())
metrics_df = pd.DataFrame(
    [
        {
            "system": "S_tuned_LoRA_mBART50_SC_to_EN",
            "BLEU": scores["BLEU"],
            "chrF": scores["chrF"],
            "train_label": "NLLB pseudo English translated from train Mandarin",
            "test_reference": TEST_EN_COL,
        }
    ]
)
display(metrics_df)

metrics_path = WORK_DIR / "lora_mbart_sc_en_metrics.csv"
preds_path = WORK_DIR / "lora_mbart_sc_en_predictions.csv"
metrics_df.to_csv(metrics_path, index=False)
test_df.to_csv(preds_path, index=False, encoding="utf-8-sig")
print("Saved metrics:", metrics_path)
print("Saved predictions:", preds_path)

,system,BLEU,chrF,train_label,test_reference
0,S_tuned_LoRA_mBART50_SC_to_EN,9.873555,30.887753,NLLB pseudo English translated from train Mand...,english


Saved metrics: /kaggle/working/lora_mbart_sc_en_metrics.csv
Saved predictions: /kaggle/working/lora_mbart_sc_en_predictions.csv


## 7. Qualitative Cases for Address Terms

In [18]:
KEY_TERMS = [
    "老汉",
    "嬢嬢",
    "幺儿",
    "幺妹",
    "婆婆",
    "老表",
    "堂客",
    "自家",
    "别个",
    "人家",
    "老师",
    "老哥",
    "美女",
    "帅哥",
    "伙计",
    "老乡",
    "老子",
    "雄起",
]

mask = test_df[TEST_SC_COL].astype(str).apply(lambda s: any(term in s for term in KEY_TERMS))
cases = test_df.loc[mask, [TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].copy()
if len(cases) == 0:
    cases = test_df[[TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].copy()

cases = cases.head(50)
cases["sentence_BLEU"] = [
    sacrebleu.sentence_bleu(pred, [ref]).score
    for pred, ref in zip(cases[PRED_EN_COL].astype(str), cases[TEST_EN_COL].astype(str))
]

cases_path = WORK_DIR / "lora_mbart_sc_en_cases.csv"
cases.to_csv(cases_path, index=False, encoding="utf-8-sig")
display(cases.head(20))
print("Saved cases:", cases_path)

,text,english,pred_english_lora_mbart,sentence_BLEU
2,谢谢谢谢李婆婆,Thank you very much 李奶奶,"Thank you, thank you.",16.233396
12,这就有点无聊了要说保卫自己的领土激战一把哎倒也无所谓了赖在人家地盘死活都不走那个鬼子们还真的...,"It's a bit boring, talking about defending one...",It's a little boring to say that defending you...,3.634081
14,啊就相当于民间的二人转的啊那么就写点这些啊比如说你要形容美女啊我以前都在说过哈长得来千娇百媚哈媚哈,This would be equivalent to the folk duo turni...,"It's like two people, so write something like ...",8.327797
27,人家弄坏了家弄坏了搞了我也看那个还,Other people messed up and I'm tired of watchi...,I'm looking at that's still going on.,4.513618
58,有天我无意翻到了格子心老师的文章写的太好了至少我写不了自愧不如他应该的确是有格子心的只是一直...,One day I accidentally came across an article ...,I didn't mean to turn to the teacher's article...,2.674468
83,还是搞成了好几对的哈因为你是个外地的是个外地所以我们来今天问一下嘛美女你们和那个电话老板聊得...,"So today we're going to ask, beautiful ladies,...","It's been a couple of years, because you're an...",12.753431
133,甜吃一口嘛别的婆娘米奇我呀黑珍珠的育儿激情而且我这个是专门买起来的哦我我不是说什么爱你专门买...,"甜食一口嘛,别的女人我不罕见,我喜欢黑珍珠的育儿激情──and I bought this ...",I don't mean love love love I say we want to h...,2.054350
166,你既应了就得耐烦些少不得替人家办办,You should have promised to be a little more p...,"You shoulder, you shoulder to endure, and you ...",2.494650


Saved cases: /kaggle/working/lora_mbart_sc_en_cases.csv


## 8. Report Wording
#
Model name:
- `S_tuned: LoRA-mBART50 end-to-end Sichuanese -> English`
#
Training data:
- Source: `train_9800_zh.csv` Sichuanese `text`
- Target: pseudo English generated by NLLB from the Mandarin pivot column
#
Evaluation:
- Test: `test_200_en.csv`
- Reference: human/gold English column
- Metrics: BLEU and chrF
#
Note:
- Because the 9800 training rows do not have human English, the training
  target is pseudo-labeled English. This should be stated clearly in the
  report.